In [1]:
# Add beh_ephys_analysis to path for imports
import os
import sys
import glob

# Get current directory
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Try to find beh_ephys_analysis
beh_ephys_path = None

if current_dir.endswith('/code') or current_dir == '/code':
    test_path = os.path.join(current_dir, 'beh_ephys_analysis')
    if os.path.exists(os.path.join(test_path, 'utils')):
        beh_ephys_path = test_path

if beh_ephys_path and os.path.exists(os.path.join(beh_ephys_path, 'utils')):
    if beh_ephys_path in sys.path:
        sys.path.remove(beh_ephys_path)
    sys.path.insert(0, beh_ephys_path)
    print(f"✓ Successfully added to path: {beh_ephys_path}")
else:
    print(f"✗ ERROR: Could not find beh_ephys_analysis/utils")
    raise ImportError("Cannot locate beh_ephys_analysis module")

# Now do the imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.beh_functions import *
from utils.pupil_utils import load_pupil
from aind_dynamic_foraging_data_utils.nwb_utils import load_nwb_from_filename
from aind_dynamic_foraging_basic_analysis.plot.plot_foraging_session import plot_foraging_session, plot_foraging_session_nwb
from aind_dynamic_foraging_basic_analysis.licks.lick_analysis import plot_lick_analysis, cal_metrics, plot_met, load_data
from harp.clock import decode_harp_clock, align_timestamps_to_anchor_points
from open_ephys.analysis import Session
import datetime
from aind_ephys_rig_qc.temporal_alignment import search_harp_line
from matplotlib.gridspec import GridSpec
import json
import itertools

import json
import numpy as np

from pynwb.file import LabMetaData
from hdmf.utils import docval

# Import through the data_management package, not flat: build_merged_nwb uses a
# relative import for its metadata helper, which only resolves when the module is
# reached via its parent package.
sys.path.insert(0, '/root/capsule/code')
from data_management.build_merged_nwb import build_combined_nwb, merge_unit_tables

print("✓ All imports successful!")
%matplotlib inline

Current directory: /code
✓ Successfully added to path: /code/beh_ephys_analysis


✓ All imports successful!


In [7]:
"""
Comprehensive test for build_combined_nwb() - validates columns, names, descriptions, and values

Checks:
1. All source columns are included with correct mapped names
2. Descriptions match column_names_description.json
3. Values match between source and combined NWB
4. Data modality tracking
"""


# Load mappings and descriptions
with open('/root/capsule/code/data_management/column_names_map.json', 'r') as f:
    COLUMN_MAP = json.load(f)
with open('/root/capsule/code/data_management/column_names_description.json', 'r') as f:
    COLUMN_DESC = json.load(f)

# Predefined NWB columns that are handled separately (not via add_unit_column)
PREDEFINED_COLS = ['spike_times', 'electrodes', 'obs_intervals', 'electrode_group']

def compare_values(source_val, nwb_val, col_name):
    """Compare two values and return if they're equal and any differences"""
    if source_val is None or (isinstance(source_val, float) and pd.isna(source_val)):
        # Source is None/NaN
        if isinstance(nwb_val, np.ndarray) and len(nwb_val) == 0:
            return True, None  # None -> empty array is OK
        if isinstance(nwb_val, float) and np.isnan(nwb_val):
            return True, None  # NaN preserved
        return False, f"Source is None but NWB is {type(nwb_val).__name__}"
    
    if isinstance(source_val, np.ndarray) and isinstance(nwb_val, np.ndarray):
        if source_val.shape != nwb_val.shape:
            return False, f"Shape mismatch: {source_val.shape} vs {nwb_val.shape}"
        if not np.allclose(source_val, nwb_val, equal_nan=True):
            return False, f"Array values differ"
        return True, None
    
    if isinstance(source_val, (int, float, np.integer, np.floating)) and isinstance(nwb_val, (int, float, np.integer, np.floating)):
        if pd.isna(source_val) and pd.isna(nwb_val):
            return True, None
        if np.isclose(source_val, nwb_val, equal_nan=True):
            return True, None
        return False, f"Value mismatch: {source_val} vs {nwb_val}"
    
    if source_val == nwb_val:
        return True, None
    
    return False, f"Value mismatch: {source_val} vs {nwb_val}"

def test_session_comprehensive(session_id, data_type='curated'):
    """Comprehensive validation of combined NWB"""
    print(f"\n{'#'*80}")
    print(f"# {session_id}")
    print(f"{'#'*80}\n")
    
    # Get source data
    session_tbl = get_session_tbl(session_id)
    merged_units = merge_unit_tables(session_id, data_type)
    
    # Build combined NWB
    save_path, nwb, modalities = build_combined_nwb(session_id, data_type, save_file=None)
    # build_combined_nwb() creates a combined NWB object in memory and returns it, so we can inspect it without saving
    # It saves the nwb only when save_file is provided
    
    if nwb is None:
        print("✗ FAILED: Could not build combined NWB")
        return None
    
    # Extract combined data
    trials_df = nwb.trials.to_dataframe() if nwb.trials else None
    units_df = nwb.units.to_dataframe() if nwb.units else None
    acq_count = len(nwb.acquisition) if nwb.acquisition else 0
    
    # # Print data modalities
    # print(f"DATA MODALITIES:")
    # print(f"{'='*80}")
    # # modality_symbols = {
    # #     'behavior_trials': '🎯',
    # #     'ephys_units': '⚡',
    # #     'lick_times': '👅',
    # #     'reward_times': '🎁',
    # #     'FP': '📊'
    # # }
    # # for mod_name, included in modalities.items():
    # #     symbol = modality_symbols.get(mod_name, '•')
    # #     status = '✓' if included else '✗'
    # #     print(f"  {status} {symbol} {mod_name}")
    
    # print(f"\nACTUAL DATA:")
    # print(f"  Trials: {len(trials_df) if trials_df is not None else 0}")
    # print(f"  Units: {len(units_df) if units_df is not None else 0}")
    # print(f"  Acquisition: {acq_count}")
    
    # Initialize summary
    summary = {
        'session': session_id,
        'modalities': modalities,
        'n_trials': len(trials_df) if trials_df is not None else 0,
        'n_units': len(units_df) if units_df is not None else 0,
        'n_acquisition': acq_count,
        'issues': {}
    }
    
    # Validate trials if they exist
    if session_tbl is not None and trials_df is not None:
        # print(f"\n{'='*80}")
        # print("TRIAL TABLE VALIDATION")
        # print(f"{'='*80}")
        
        trial_map = COLUMN_MAP['behavior_trial_columns']
        trial_desc = COLUMN_DESC['behavior_trial_columns']
        
        missing_cols = []
        wrong_descriptions = []
        value_mismatches = []
        
        for orig_col, mapped_name in trial_map.items():
            if orig_col not in session_tbl.columns:
                continue
            
            if mapped_name not in trials_df.columns:
                missing_cols.append(f"{orig_col} -> {mapped_name}")
                continue
            
            # Check description
            col_idx = trials_df.columns.get_loc(mapped_name)
            actual_desc = nwb.trials.columns[col_idx].description
            expected_desc = trial_desc.get(orig_col, f'Trial column: {orig_col}')
            
            if actual_desc != expected_desc:
                wrong_descriptions.append(f"{mapped_name}")
            
            # Check values (sample first 5 rows)
            for idx in range(min(5, len(session_tbl))):
                source_val = session_tbl[orig_col].iloc[idx]
                nwb_val = trials_df[mapped_name].iloc[idx]
                
                is_equal, diff = compare_values(source_val, nwb_val, mapped_name)
                if not is_equal:
                    value_mismatches.append(f"{mapped_name} row {idx}: {diff}")
                    break
        
        # print(f"Source columns: {len([c for c in trial_map.keys() if c in session_tbl.columns])}")
        # print(f"NWB columns: {len(trials_df.columns)}")
        
        # if not missing_cols:
        #     print(f"✓ All source trial columns included")
        # else:
        #     print(f"✗ Missing columns: {len(missing_cols)}")
            
        # if not wrong_descriptions:
        #     print(f"✓ All trial descriptions match")
        # else:
        #     print(f"⚠ Wrong descriptions: {len(wrong_descriptions)}")
            
        # if not value_mismatches:
        #     print(f"✓ Trial values match (sampled first 5 rows)")
        # else:
        #     print(f"✗ Value mismatches: {len(value_mismatches)}")
        
        summary['issues']['trials'] = {
            'missing_cols': len(missing_cols),
            'wrong_desc': len(wrong_descriptions),
            'value_mismatch': len(value_mismatches)
        }
    
    # Validate units if they exist
    if merged_units is not None and units_df is not None:
        # print(f"\n{'='*80}")
        # print("UNIT TABLE VALIDATION")
        # print(f"{'='*80}")
        
        unit_map_custom = COLUMN_MAP['unit_columns_custom']
        unit_map_ks = COLUMN_MAP['unit_columns_ks']
        
        missing_unit_cols = []
        
        # Check custom columns (excluding predefined NWB columns)
        for orig_col, mapped_name in unit_map_custom.items():
            if orig_col not in merged_units.columns:
                continue
            if 'duplicate as' in mapped_name:
                continue
            if 'similar to' in mapped_name:
                mapped_name = mapped_name.split(';')[0].strip()
            
            # Skip predefined columns - they're handled separately
            if mapped_name in PREDEFINED_COLS:
                continue
            
            if mapped_name not in units_df.columns:
                missing_unit_cols.append(f"{orig_col} -> {mapped_name}")
        
        # Check KS columns (excluding predefined NWB columns)
        for orig_col, mapped_name in unit_map_ks.items():
            if orig_col not in merged_units.columns:
                continue
            
            # Skip predefined columns - they're handled separately
            if mapped_name in PREDEFINED_COLS:
                continue
            
            if mapped_name not in units_df.columns:
                missing_unit_cols.append(f"{orig_col} -> {mapped_name}")
        
        # print(f"Source columns: {len(merged_units.columns)}")
        # print(f"NWB columns: {len(units_df.columns)}")
        
        # if not missing_unit_cols:
        #     print(f"✓ All source unit columns included (excluding predefined NWB columns)")
        # else:
        #     print(f"✗ Missing unit columns: {len(missing_unit_cols)}")
        #     for col in missing_unit_cols[:5]:
        #         print(f"    - {col}")
        
        summary['issues']['units'] = {
            'missing_cols': len(missing_unit_cols)
        }
    
    # # Summary
    # print(f"\n{'='*80}")
    # print("SUMMARY")
    # print(f"{'='*80}")
    
    total_issues = sum(sum(v.values()) if isinstance(v, dict) else 0 
                      for v in summary['issues'].values())
    
    # if total_issues == 0:
    #     print(f"✓ ALL CHECKS PASSED")
    #     summary['status'] = 'PASSED'
    # else:
    #     print(f"⚠ Total issues: {total_issues}")
    #     summary['status'] = 'ISSUES'
    
    # Print summary dictionary
    # print(f"\nSUMMARY DICT:")
    # print(json.dumps(summary, indent=2, default=str))
    
    return summary


In [6]:

# Test sessions (including one with FP data)
example_sessions = [
    'behavior_754897_2025-03-13_11-20-42',
    'behavior_ZS061_2021-04-08_18-01-30',
    'behavior_754898_2025-01-01_20-40-03',  # Has FP data
]

all_summaries = []
for session in example_sessions:
    try:
        summary = test_session_comprehensive(session, data_type='curated')
        if summary:
            all_summaries.append(summary)
    except Exception as e:
        print(f"✗ Exception: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\n{'#'*80}")
print("ALL SESSIONS SUMMARY")
print(f"{'#'*80}\n")
print(json.dumps(all_summaries, indent=2, default=str))


################################################################################
# behavior_754897_2025-03-13_11-20-42
################################################################################

No pupil file found

################################################################################
# behavior_ZS061_2021-04-08_18-01-30
################################################################################

✗ Exception: nothing found at path ''

################################################################################
# behavior_754898_2025-01-01_20-40-03
################################################################################



Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py", line 566, in __open_file_consolidated
    return zarr.open_consolidated(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/zarr/convenience.py", line 1362, in open_consolidated
    meta_store = ConsolidatedStoreClass(store, metadata_key=metadata_key)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/zarr/storage.py", line 3045, in __init__
    meta = json_loads(self.store[metadata_key])
                      ~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/zarr/storage.py", line 1120, in __getitem__
    raise KeyError(key)
KeyError: '.zmetadata'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_464005/1466519344.py", line 11, in <module>
    summary = test_session_comprehensive(

✗ Exception: nothing found at path ''


################################################################################
ALL SESSIONS SUMMARY
################################################################################

[
  {
    "session": "behavior_754897_2025-03-13_11-20-42",
    "modalities": {
      "behavior_trials": true,
      "ephys_units": true,
      "lick_times": true,
      "reward_times": true,
      "FP": false,
      "pupil": false,
      "tongue_movements": false,
      "keypoint_tracking": true,
      "aind_metadata": false,
      "beh_version": "processed",
      "nwb_created": "2026-08-18T16:23:56.436146+00:00",
      "nwb_saved": null
    },
    "n_trials": 564,
    "n_units": 269,
    "n_acquisition": 4,
    "issues": {
      "trials": {
        "missing_cols": 0,
        "wrong_desc": 5,
        "value_mismatch": 2
      },
      "units": {
        "missing_cols": 0
      }
    }
  }
]


Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py", line 566, in __open_file_consolidated
    return zarr.open_consolidated(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/zarr/convenience.py", line 1362, in open_consolidated
    meta_store = ConsolidatedStoreClass(store, metadata_key=metadata_key)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/zarr/storage.py", line 3045, in __init__
    meta = json_loads(self.store[metadata_key])
                      ~~~~~~~~~~^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/zarr/storage.py", line 1120, in __getitem__
    raise KeyError(key)
KeyError: '.zmetadata'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_464005/1466519344.py", line 11, in <module>
    summary = test_session_comprehensive(

In [4]:
# make a table of the results, each key in the summary dict becomes a column, and each session is a row
summary_df = pd.DataFrame(all_summaries)
# put modalities into separate columns
modalities_df = summary_df['modalities'].apply(pd.Series)
# put issues into separate columns
issues_df = summary_df['issues'].apply(lambda x: pd.Series({
    'trials_missing_cols': x['trials']['missing_cols'] if 'trials' in x else np.nan,
    'trials_wrong_desc': x['trials']['wrong_desc'] if 'trials' in x else np.nan,
    'trials_value_mismatch': x['trials']['value_mismatch'] if 'trials' in x else np.nan,
    'units_missing_cols': x['units']['missing_cols'] if 'units' in x else np.nan,
}))
# merge with summary_df
final_df = pd.concat([summary_df.drop(columns=['modalities']), modalities_df], axis=1)
final_df = pd.concat([final_df.drop(columns=['issues']), issues_df], axis=1)
print(final_df)

                               session  n_trials  n_units  n_acquisition  \
0  behavior_754897_2025-03-13_11-20-42       564      269              4   
1   behavior_ZS061_2021-04-08_18-01-30       386        2              4   
2  behavior_754898_2025-01-01_20-40-03       542        0             53   

   behavior_trials  ephys_units  lick_times  reward_times     FP  pupil  \
0             True         True        True          True  False  False   
1             True         True        True          True  False   True   
2             True        False        True          True   True  False   

   lick_video  keypoint_tracking beh_version  \
0       False               True   processed   
1       False              False         raw   
2       False              False         raw   

                        nwb_created nwb_saved  trials_missing_cols  \
0  2026-08-12T19:56:15.634898+00:00      None                  0.0   
1  2026-08-12T19:56:50.307020+00:00      None                

In [10]:
# load all used sessions
dfs = [pd.read_csv(CAPSULE_ROOT + '/code/data_management/session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_session_assets.csv'),
        pd.read_csv(CAPSULE_ROOT + '/code/data_management/hopkins_FP_session_assets.csv')]
df = pd.concat(dfs)
# remove empty rows
df = df.dropna(subset=['session_id'])
session_list = df['session_id'].values.tolist()

## Test

In [2]:
example_sessions = [
    'behavior_ZS062_2021-05-06_15-46-14',
    'behavior_ZS059_2021-04-29_14-02-45',
    'behavior_ZS061_2021-04-08_18-01-30',
    'behavior_781166_2025-05-13_14-04-27',
    'behavior_754897_2025-03-12_12-23-15',
    'behavior_754897_2025-03-13_11-20-42',
    'behavior_754898_2025-01-01_20-40-03',
    'behavior_749472_2025-01-09_13-56-02',
    'behavior_754896_2025-01-03_17-20-19'
]

In [2]:
results_modality_csv = '/root/capsule/data/LC_all_nwbs/modalities_df.csv'
example_sessions = pd.read_csv(results_modality_csv)
example_sessions = example_sessions[example_sessions['error']].session.to_list()

In [3]:
from joblib import Parallel, delayed
save_dir = '/root/capsule/scratch/combined/nwb_testing_folder/'
def build_one(session, **kwargs):
    try:
        save_file = os.path.join(save_dir, f"{session}_combined.nwb")
        nwb_path, _, modalities = build_combined_nwb(
            session,
            'curated',
            save_file=save_file,
            **kwargs,
        )

        return {
            'session': session,
            'error': False,
            'error_msg': None,
            'nwb_path': nwb_path,
            **modalities,
        }

    except Exception as e:
        return {
            'session': session,
            'error': True,
            'error_msg': f"{type(e).__name__}: {e}",
            'nwb_path': None,
        }


In [2]:
nwb_path, _, modalities = build_combined_nwb(
    'behavior_669489_2023-07-04_14-24-45',
    'curated',
    save_file=None,
    add_metadata=True,
)

No custom unit table found for behavior_669489_2023-07-04_14-24-45
No unit tables to merge for behavior_669489_2023-07-04_14-24-45 - will create NWB with behavior/acquisition only


No unit table found for behavior_669489_2023-07-04_14-24-45 in curated data.
No pupil file found.
Output file behavior_669489_2023-07-04_14-24-45/ephys/opto/curated/drift does not exist, skipping neuron_drift_curation
Output file behavior_669489_2023-07-04_14-24-45/ephys/opto/curated/behavior_669489_2023-07-04_14-24-45_antidromic_results.pkl does not exist, skipping antidromic_tagging
Output file behavior_669489_2023-07-04_14-24-45/ephys/opto/curated/figures does not exist, skipping opto_tagging
Output file behavior_669489_2023-07-04_14-24-45/ephys/opto/curated/behavior_669489_2023-07-04_14-24-45_opto_sigs.pkl does not exist, skipping opto_tagging_significance
Output file behavior_669489_2023-07-04_14-24-45/ephys/opto/curated/behavior_669489_2023-07-04_14-24-45_curated_soma_opto_tagging_summary.pkl does not exist, skipping opto_tagging_summary
Output file behavior_669489_2023-07-04_14-24-45/ephys/curated/processed/ccf_unit_locations.csv does not exist, skipping compute_neuron_locations

ERROR:aind_data_schema.utils.compatibility_check:Active devices {'Fiber 1', 'Fiber 0', 'Fiber 2'} were not found in Instrument.components. Note: These devices may be valid if they exist in Procedures.
ERROR:aind_data_schema.utils.compatibility_check:Active devices {'Fiber 1', 'Fiber 0', 'Fiber 2'} were not found in Instrument.components. Note: These devices may be valid if they exist in Procedures.


In [4]:
# n_jobs is memory-bound (each worker holds a whole NWB in memory), not CPU-bound.
# Worker prints don't reach the notebook: progress comes from verbose=, failures
# come back in the error_msg column.
parameters = {"add_metadata": True}
modalities_combined = Parallel(n_jobs=8, backend='loky', verbose=10)(
    delayed(build_one)(session, **parameters) for session in example_sessions
)

# last store written, for the loading test below
write_save_file = next((r['nwb_path'] for r in reversed(modalities_combined) if r['nwb_path']), None)
print(f"{sum(not r['error'] for r in modalities_combined)}/{len(modalities_combined)} sessions written")
for r in modalities_combined:
    if r['error']:
        print(f"  ✗ {r['session']}: {r['error_msg']}")


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


No unit table found for behavior_744779_2024-11-15_12-54-11 in curated data.
No unit table found for behavior_744779_2024-11-13_13-35-38 in curated data.
No pupil file found.
No pupil file found.
No unit table found for behavior_669489_2023-06-26_16-23-02 in curated data.


No custom unit table found for behavior_744779_2024-11-15_12-54-11
No custom unit table found for behavior_744779_2024-11-13_13-35-38
No unit tables to merge for behavior_744779_2024-11-15_12-54-11 - will create NWB with behavior/acquisition only
No unit tables to merge for behavior_744779_2024-11-13_13-35-38 - will create NWB with behavior/acquisition only
No session table found for behavior_744779_2024-11-13_13-35-38 - will create NWB without behavior trials
No session table found for behavior_744779_2024-11-15_12-54-11 - will create NWB without behavior trials
Behavior NWB not found at /root/capsule/data/scratch_data/744779/behavior_744779_2024-11-13_13-35-38/behavior/behavior_744779_2024-11-13_13-35-38.nwb
Behavior NWB not found at /root/capsule/data/scratch_data/744779/behavior_744779_2024-11-15_12-54-11/behavior/behavior_744779_2024-11-15_12-54-11.nwb
No custom unit table found for behavior_669489_2023-06-26_16-23-02
No unit tables to merge for behavior_669489_2023-06-26_16-23-02

No unit table found for behavior_744779_2024-11-12_13-10-14 in curated data.
No pupil file found.


No custom unit table found for behavior_744779_2024-11-12_13-10-14
No unit tables to merge for behavior_744779_2024-11-12_13-10-14 - will create NWB with behavior/acquisition only
No session table found for behavior_744779_2024-11-12_13-10-14 - will create NWB without behavior trials
Behavior NWB not found at /root/capsule/data/scratch_data/744779/behavior_744779_2024-11-12_13-10-14/behavior/behavior_744779_2024-11-12_13-10-14.nwb


...

specimen_procedures.1
  Assertion failed, FluorescentStain or ProbeReagent required if procedure_type is Immunolabeling. [type=assertion_error, input_value={'object_type': 'Specimen...ils': [], 'notes': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/assertion_error
specimen_procedures.2
  Assertion failed, FluorescentStain or ProbeReagent required if procedure_type is Immunolabeling. [type=assertion_error, input_value={'object_type': 'Specimen...ils': [], 'notes': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/assertion_error
/opt/conda/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `Surgery` - serialized value may not be as expected [input_value={'object_type': 'Surgery'...9624']}], 'notes': None}, input_type=dict])
  PydanticSerializationUnexpectedValue(Expected `Injection` - serialized value may not

Failed to validate procedures to capture procedure device names: 2 validation errors for Procedures
specimen_procedures.1
  Assertion failed, FluorescentStain or ProbeReagent required if procedure_type is Immunolabeling. [type=assertion_error, input_value={'object_type': 'Specimen...ils': [], 'notes': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/assertion_error
specimen_procedures.2
  Assertion failed, FluorescentStain or ProbeReagent required if procedure_type is Immunolabeling. [type=assertion_error, input_value={'object_type': 'Specimen...ils': [], 'notes': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/assertion_error


[Parallel(n_jobs=8)]: Done 105 out of 109 | elapsed: 19.1min remaining:   43.7s


109/109 sessions written


[Parallel(n_jobs=8)]: Done 109 out of 109 | elapsed: 19.6min finished


In [5]:
# one row per session, one column per modality field
modalities_df = pd.DataFrame(modalities_combined)
lead_cols = ['session', 'error', 'error_msg', 'nwb_path']
modalities_df = modalities_df[lead_cols + [c for c in modalities_df.columns if c not in lead_cols]]

In [6]:
modalities_df.to_csv(f"{save_dir}/errors_metadata.csv")

In [20]:
modalities_df

,session,error,error_msg,nwb_path,behavior_trials,ephys_units,lick_times,reward_times,FP,pupil,tongue_movements,keypoint_tracking,aind_metadata,beh_version,ephys_version,nwb_created,nwb_saved
0,behavior_744779_2024-11-12_13-10-14,False,None,no valid data,False,False,False,False,False,False,False,False,False,none,none,2026-09-15T22:15:58.107887+00:00,None
1,behavior_744779_2024-11-13_13-35-38,False,None,no valid data,False,False,False,False,False,False,False,False,False,none,none,2026-09-15T22:15:58.074780+00:00,None
2,behavior_744779_2024-11-15_12-54-11,False,None,no valid data,False,False,False,False,False,False,False,False,False,none,none,2026-09-15T22:15:58.142330+00:00,None
3,behavior_751004_2024-12-21_13-28-28,False,None,/root/capsule/scratch/combined/nwb_testing_fol...,True,True,True,True,False,False,True,True,False,processed,curated,2026-09-15T22:16:12.341597+00:00,2026-09-15T22:16:34.814261+00:00
4,behavior_751766_2025-02-15_12-08-11,False,None,/root/capsule/scratch/combined/nwb_testing_fol...,True,False,True,True,False,False,True,True,False,processed,none,2026-09-15T22:15:59.324545+00:00,2026-09-15T22:16:21.230753+00:00
5,behavior_758018_2025-03-20_11-53-05,False,None,/root/capsule/scratch/combined/nwb_testing_fol...,True,True,True,True,False,False,True,True,False,processed,curated,2026-09-15T22:16:12.174677+00:00,2026-09-15T22:16:31.500600+00:00


## Test loading

In [8]:
write_save_file = '/root/capsule/scratch/combined/nwb_testing_folder/behavior_754897_2025-03-12_12-23-15_combined.nwb.zarr'

In [9]:
io = NWBZarrIO(write_save_file, 'r', load_namespaces=True)
nwb_loaded = io.read()
# ... use nwb_loaded ...
io.close()

In [10]:
unit_tbl = nwb_loaded.units.to_dataframe()

In [11]:
# Read the combined JSON metadata using ONLY hdmf_zarr / NWBZarrIO.
# On-disk, a LabMetaData container lives at /general/<name>, NOT /general/lab_meta_data/<name>.
with NWBZarrIO(write_save_file, 'r') as io_r:
    raw = io_r.file['general/aind_metadata/json_data'][()]

if isinstance(raw, (bytes, np.bytes_)):
    raw = raw.decode()
elif isinstance(raw, np.ndarray):
    raw = raw.item()
    if isinstance(raw, bytes):
        raw = raw.decode()

all_meta = json.loads(str(raw))
print('files:', list(all_meta.keys()))
print('acquisition subject_id:', all_meta['acquisition'].get('subject_id'))

KeyError: 'general/aind_metadata/json_data'